## 2.1 卷积层理论计算

输入一张大小为 $3 \times 32 \times 32$ （通道数 $\times$ 高 $\times$ 宽）的彩色图像。通过一个卷积层，该层包含 16 个卷积核，每个卷积核的大小为 $3 \times 5 \times 5$ 。设定填充（Padding）为 2，步幅（Stride）为 2。

### 1. 输出特征图尺寸

高度/宽度计算公式：

$$H_{out} = \left\lfloor \frac{H_{in} + 2 \times \text{padding} - \text{kernel\_size}}{\text{stride}} \right\rfloor + 1$$

代入数值：

$$H_{out} = \left\lfloor \frac{32 + 4 - 5}{2} \right\rfloor + 1 = \left\lfloor \frac{31}{2} \right\rfloor + 1 = 15 + 1 = 16$$

输出尺寸为：**$16 \times 16 \times 16$**（通道数 $\times$ 高 $\times$ 宽）

### 2. 单个输出通道的一个像素值的点乘次数

每个卷积核大小为 $3 \times 5 \times 5 = 75$，因此单个输出像素需要 **75 次乘法**。

In [16]:
import numpy as np

def max_pool2d_manual(input, kernel_size, stride=1, padding=0):
    """二维最大池化前向传播"""
    if isinstance(kernel_size, int):
        kh = kw = kernel_size
    else:
        kh, kw = kernel_size
    if isinstance(stride, int):
        sh = sw = stride
    else:
        sh, sw = stride
    if isinstance(padding, int):
        ph = pw = padding
    else:
        ph, pw = padding

    C, H, W = input.shape
    padded = np.pad(input, ((0, 0), (ph, ph), (pw, pw)), mode='constant')
    H_out = (H + 2*ph - kh) // sh + 1
    W_out = (W + 2*pw - kw) // sw + 1

    output = np.zeros((C, H_out, W_out))
    for c in range(C):
        for i in range(H_out):
            for j in range(W_out):
                h_start = i * sh
                h_end = h_start + kh
                w_start = j * sw
                w_end = w_start + kw
                output[c, i, j] = np.max(padded[c, h_start:h_end, w_start:w_end])
    return output

#测试代码
if __name__ == "__main__":
    test_input = np.array([[
        [1, 2, 3, 4],
        [5, 6, 7, 8],
        [9, 10, 11, 12],
        [13, 14, 15, 16]
    ]], dtype=np.float32)
    print("=" * 50)
    print("2.2 最大池化测试")
    print("=" * 50)
    print(f"输入形状: {test_input.shape}")
    print(f"输入内容:\n{test_input[0]}\n")
    
    output = max_pool2d_manual(test_input, kernel_size=2, stride=2, padding=0)
    print(f"池化后形状: {output.shape}")
    print(f"池化结果:\n{output[0]}\n")

2.2 最大池化测试
输入形状: (1, 4, 4)
输入内容:
[[ 1.  2.  3.  4.]
 [ 5.  6.  7.  8.]
 [ 9. 10. 11. 12.]
 [13. 14. 15. 16.]]

池化后形状: (1, 2, 2)
池化结果:
[[ 6.  8.]
 [14. 16.]]



## 3.1 VGG 卷积核参数量计算

假设输入和输出的特征图通道数均为 $C$。

### 1. 单个 $5 \times 5$ 卷积层（不带偏置）的参数量

$$5 \times 5 \times C \times C = 25C^2$$

### 2. 两个串联的 $3 \times 3$ 卷积层（不带偏置，两层通道数都为 $C$）的总参数量

单个 $3 \times 3$ 卷积参数量：$3 \times 3 \times C \times C = 9C^2$

两个串联的总参数量：$2 \times 9C^2 = 18C^2$

In [17]:
import torch
import torch.nn as nn

class NiNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        super(NiNBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU()
        )
    def forward(self, x):
        return self.block(x)

#测试代码
if __name__ == "__main__":
    print("=" * 50)
    print("3.2 NiN块测试")
    print("=" * 50)
    
    nin_block = NiNBlock(in_channels=3, out_channels=96, kernel_size=5, stride=1, padding=2)
    print(f"NiN块结构:\n{nin_block}\n")
    
    test_input = torch.randn(1, 3, 32, 32)
    print(f"输入形状: {test_input.shape}")
    
    # 前向传播
    output = nin_block(test_input)
    print(f"输出形状: {output.shape}")
    print(f"输出通道数: {output.shape[1]}")
    print(f"输出高度/宽度: {output.shape[2]} x {output.shape[3]}\n")

3.2 NiN块测试
NiN块结构:
NiNBlock(
  (block): Sequential(
    (0): Conv2d(3, 96, kernel_size=(5, 5), stride=(1, 1), padding=(2, 2))
    (1): ReLU()
    (2): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1))
    (3): ReLU()
    (4): Conv2d(96, 96, kernel_size=(1, 1), stride=(1, 1))
    (5): ReLU()
  )
)

输入形状: torch.Size([1, 3, 32, 32])
输出形状: torch.Size([1, 96, 32, 32])
输出通道数: 96
输出高度/宽度: 32 x 32



## 4.1 批量归一化计算

在一个小批量训练中，某一个通道内某一特定空间位置的特征值在 4 个样本上的输出分别为：$x_1 = 2, x_2 = 4, x_3 = 6, x_4 = 8$。假设当前批量归一化层学到的缩放参数 $\gamma = 2$，平移参数 $\beta = 1$，常数 $\epsilon = 0$。

### 步骤

1. 计算均值：

$$\mu = \frac{2 + 4 + 6 + 8}{4} = 5$$

2. 计算方差：

$$\sigma^2 = \frac{(2-5)^2 + (4-5)^2 + (6-5)^2 + (8-5)^2}{4} = \frac{9 + 1 + 1 + 9}{4} = 5$$

3. 标准化：

$$\hat{x}_i = \frac{x_i - \mu}{\sqrt{\sigma^2 + \epsilon}}$$

$$\hat{x}_1 = \frac{-3}{\sqrt{5}},\quad \hat{x}_2 = \frac{-1}{\sqrt{5}},\quad \hat{x}_3 = \frac{1}{\sqrt{5}},\quad \hat{x}_4 = \frac{3}{\sqrt{5}}$$

4. 输出：

$$y_i = \gamma \hat{x}_i + \beta = 2\hat{x}_i + 1$$

### 最终结果

$$y_1 = 1 - \frac{6}{\sqrt{5}},\quad y_2 = 1 - \frac{2}{\sqrt{5}},\quad y_3 = 1 + \frac{2}{\sqrt{5}},\quad y_4 = 1 + \frac{6}{\sqrt{5}}$$

In [18]:
import torch
import torch.nn as nn

class Residual(nn.Module):
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        super(Residual, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, stride=stride)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()

        if use_1x1conv:
            self.shortcut = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride)
        else:
            self.shortcut = None

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.shortcut:
            x = self.shortcut(x)
        out += x
        return self.relu(out)

#测试代码
if __name__ == "__main__":
    print("=" * 50)
    print("4.2 残差块测试")
    print("=" * 50)
    
    # 测试1: 输入输出通道相同，不用1x1卷积
    block1 = Residual(in_channels=64, out_channels=64, use_1x1conv=False)
    print(f"残差块1 (无1x1卷积):")
    print(f"  - 输入通道: 64, 输出通道: 64")
    
    test_input = torch.randn(1, 64, 32, 32)
    output1 = block1(test_input)
    print(f"  - 输入形状: {test_input.shape}")
    print(f"  - 输出形状: {output1.shape}\n")
    
    # 测试2: 输入输出通道不同，使用1x1卷积
    block2 = Residual(in_channels=32, out_channels=64, use_1x1conv=True, stride=2)
    print(f"残差块2 (带1x1卷积, stride=2):")
    print(f"  - 输入通道: 32, 输出通道: 64")
    
    test_input2 = torch.randn(1, 32, 32, 32)
    output2 = block2(test_input2)
    print(f"  - 输入形状: {test_input2.shape}")
    print(f"  - 输出形状: {output2.shape}")
    print(f"  - 注意: stride=2使空间尺寸减半为 {output2.shape[2]}x{output2.shape[3]}\n")

4.2 残差块测试
残差块1 (无1x1卷积):
  - 输入通道: 64, 输出通道: 64
  - 输入形状: torch.Size([1, 64, 32, 32])
  - 输出形状: torch.Size([1, 64, 32, 32])

残差块2 (带1x1卷积, stride=2):
  - 输入通道: 32, 输出通道: 64
  - 输入形状: torch.Size([1, 32, 32, 32])
  - 输出形状: torch.Size([1, 64, 16, 16])
  - 注意: stride=2使空间尺寸减半为 16x16



## 5.1 微调理论问题

### 1. 为什么底层学习率小，顶层学习率大？

- **底层特征提取层**学习的是通用的低级特征（如边缘、纹理），在源数据集上已经训练得很好，适用于大多数任务，因此应保留预训练知识，设置较小的学习率甚至冻结。
- **顶层输出层**是任务相关的，需要快速适应新数据集，因此设置较大的学习率。

### 2. 目标数据集很小且与源数据集相似时，如何防止过拟合？

- 冻结大部分底层特征提取层，只微调顶层分类器。
- 使用较小的学习率。
- 采用更强的正则化（如 Dropout、权重衰减）。
- 使用数据增广。

In [5]:
import torchvision.transforms as transforms

# 随机裁剪：面积比例 0.08~1.0，裁剪后缩放到 224x224
transform_crop = transforms.RandomResizedCrop(224, scale=(0.08, 1.0))

In [6]:
transform_flip = transforms.RandomHorizontalFlip(p=0.5)

In [7]:
transform_color = transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5)

In [8]:
transform_tensor = transforms.ToTensor()

In [19]:
import torchvision.transforms as transforms
from PIL import Image
import numpy as np

# 创建增广管道
augmentation_pipeline = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    transforms.ToTensor()
])

#测试代码
if __name__ == "__main__":
    print("=" * 50)
    print("5.2 图像增广测试")
    print("=" * 50)
    
    # 创建模拟图像
    mock_image_array = np.random.randint(0, 255, (300, 300, 3), dtype=np.uint8)
    mock_image = Image.fromarray(mock_image_array)
    print(f"原始图像尺寸: {mock_image.size}")
    print(f"原始图像模式: {mock_image.mode}")
    
    # 应用增广
    print("\n应用增广管道（运行5次展示随机性）:")
    for i in range(5):
        augmented = augmentation_pipeline(mock_image)
        print(f"  第{i+1}次 - 输出形状: {augmented.shape}, 像素范围: [{augmented.min():.3f}, {augmented.max():.3f}]")
    
    print("\n增广管道包含的变换:")
    print("  1. RandomResizedCrop(224, scale=(0.08, 1.0)) - 随机裁剪并缩放到224x224")
    print("  2. RandomHorizontalFlip(p=0.5) - 50%概率水平翻转")
    print("  3. ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5) - 颜色抖动")
    print("  4. ToTensor() - 转换为张量并归一化到[0,1]")

5.2 图像增广测试
原始图像尺寸: (300, 300)
原始图像模式: RGB

应用增广管道（运行5次展示随机性）:
  第1次 - 输出形状: torch.Size([3, 224, 224]), 像素范围: [0.000, 1.000]
  第2次 - 输出形状: torch.Size([3, 224, 224]), 像素范围: [0.000, 0.714]
  第3次 - 输出形状: torch.Size([3, 224, 224]), 像素范围: [0.008, 0.588]
  第4次 - 输出形状: torch.Size([3, 224, 224]), 像素范围: [0.114, 0.976]
  第5次 - 输出形状: torch.Size([3, 224, 224]), 像素范围: [0.114, 0.494]

增广管道包含的变换:
  1. RandomResizedCrop(224, scale=(0.08, 1.0)) - 随机裁剪并缩放到224x224
  2. RandomHorizontalFlip(p=0.5) - 50%概率水平翻转
  3. ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5) - 颜色抖动
  4. ToTensor() - 转换为张量并归一化到[0,1]


## 6.1 交并比（IoU）计算

真实框 $A = [10, 10, 50, 50]$，预测框 $B = [30, 30, 70, 70]$

### 1. 交集面积

交集左上角：$(\max(10,30), \max(10,30)) = (30, 30)$

交集右下角：$(\min(50,70), \min(50,70)) = (50, 50)$

交集面积：$(50 - 30) \times (50 - 30) = 20 \times 20 = 400$

### 2. 并集面积

A 面积：$(50 - 10)^2 = 1600$

B 面积：$(70 - 30)^2 = 1600$

并集面积：$1600 + 1600 - 400 = 2800$

### 3. IoU

$$\text{IoU} = \frac{400}{2800} = \frac{1}{7} \approx 0.142857$$

In [20]:
import torch
import torch.nn.functional as F

def label_smoothing_cross_entropy(logits, labels, epsilon=0.1):
    """标签平滑交叉熵损失"""
    K = logits.size(-1)
    log_probs = F.log_softmax(logits, dim=-1)
    
    with torch.no_grad():
        true_dist = torch.zeros_like(log_probs).fill_(epsilon / (K - 1))
        true_dist.scatter_(1, labels.unsqueeze(1), 1 - epsilon)
    
    return torch.mean(torch.sum(-true_dist * log_probs, dim=-1))

#测试代码
if __name__ == "__main__":
    print("=" * 50)
    print("6.2 标签平滑交叉熵测试")
    print("=" * 50)
    
    num_classes = 5
    epsilon = 0.1
    batch_size = 3
    
    torch.manual_seed(42)  
    logits = torch.randn(batch_size, num_classes)
    labels = torch.tensor([0, 2, 4]) 
    
    print(f"类别数 K = {num_classes}")
    print(f"平滑因子 ε = {epsilon}")
    print(f"批次大小 = {batch_size}\n")
    
    print("模型输出 logits:")
    print(logits)
    print(f"\n真实标签: {labels.tolist()}\n")
    
    ce_loss = F.cross_entropy(logits, labels)
    print(f"普通交叉熵损失: {ce_loss.item():.6f}")
    
    ls_loss = label_smoothing_cross_entropy(logits, labels, epsilon)
    print(f"标签平滑交叉熵损失: {ls_loss.item():.6f}")
    
    print(f"\n标签平滑目标分布（每个样本）:")
    K = num_classes
    smooth_positive = 1 - epsilon
    smooth_negative = epsilon / (K - 1)
    print(f"  - 真实类别概率: {smooth_positive}")
    print(f"  - 其他类别概率: {smooth_negative:.4f}")

6.2 标签平滑交叉熵测试
类别数 K = 5
平滑因子 ε = 0.1
批次大小 = 3

模型输出 logits:
tensor([[ 0.3367,  0.1288,  0.2345,  0.2303, -1.1229],
        [-0.1863,  2.2082, -0.6380,  0.4617,  0.2674],
        [ 0.5349,  0.8094,  1.1103, -1.6898, -0.9890]])

真实标签: [0, 2, 4]

普通交叉熵损失: 2.528892
标签平滑交叉熵损失: 2.460996

标签平滑目标分布（每个样本）:
  - 真实类别概率: 0.9
  - 其他类别概率: 0.0250
